# Crust failure rate interpolator

This notebook creates an interpolator function for the evolution of the crust failure rate due to magnetic stresses as a function of time, obtained from magneto-thermal simulations for different values of the initial magnetic field.
For more details on the implementation of the crust failures in the magneto-thermal 2D code see [Dehman et al. 2020](https://ui.adsabs.harvard.edu/abs/2020ApJ...902L..32D/abstract).
The interpolator serves as a function that, given the age and the initial magnetic field of a neutron star, gives as output the failure rate at that age. 
The original files contains the information on the time each failure happens and the released magnetic energy. 
Depending on the initial magnetic field configuration and magnetic energy in the crust, failure events might not occur below an initial polar dipolar magnetic field value as the magnetic stresses are not strong enough to cause any failure.
Since we have only sets of failures for initial magnetic field values of $10^{12}$, $10^{13}$, $10^{14}$, $10^{15}$ and $5 \times 10^{15}$ G, we need to interpolate between them to obtain the expected failures rate for any given age and initial magnetic field.

Note that the failure rate estimated from the magneto-thermal model is computed by assuming that the mechanic stresses in the crust are reset after every failure event, while the magnetic stresses continue to build up, i.e., the magnetic field remain tangled (see [Dehman et al. 2020](https://ui.adsabs.harvard.edu/abs/2020ApJ...902L..32D/abstract) for more details). This might lead to an overestimate of the number of failure events.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pathlib
import pickle
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import make_interp_spline
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg

In [ ]:
def failure_rate_fit(
    rate_initial: float,
    t: np.ndarray,
    tau: float,
    index: float,
) -> np.ndarray:
    """
    An analytical function to fit the failure rate as a function of time.

    Args:
        rate_initial (float): Initial failure rate as number of events per year.
        t (np.ndarray): Time in [yr].
        tau (float): Timescale in [yr] when to start the power-law decay.
        index (float): Power-law index for the decay in failure rate in time.

    Returns:
        (np.ndarray): failure rate as a function of time t.
    """

    rate = rate_initial * (1 + t / tau) ** index

    return rate

## Load the results from the magneto-thermal simulations

In [ ]:
base_path = pathlib.Path("../../")

if cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e12_H.d")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e13_H.d")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e14_H.d")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e15_H.d")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_5e15_H.d")    
elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_light":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e12_L.d")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e13_L.d")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e14_L.d")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_1e15_L.d")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_50-50_5e15_L.d")
elif cfg["magneto-thermal_model"] == "BSk24_multi_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e12_H.d")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e13_H.d")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e14_H.d")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e15_H.d")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_5e15_H.d")
elif cfg["magneto-thermal_model"] == "BSk24_multi_light":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e12_L.d")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e13_L.d")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e14_L.d")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_1e15_L.d")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "failures_multi_5e15_L.d")
else:
    raise ValueError("The provided magneto-thermal model does not have information on the crust failures.")

In [ ]:
# Define file paths in a dictionary.
file_paths = {
    '1e12': simB12_path,
    '1e13': simB13_path,
    '1e14': simB14_path,
    '1e15': simB15_path,
    '5e15': simB5e15_path
}

# Define column names.
columns = ['time', 'energy', 'theta', 'radius', 'volume', 'timescale']

# Load DataFrames safely into a dictionary.
dfs = {}
for key, path in file_paths.items():
    # Check if the files are empty, i.e., contain no failure events.
    if os.path.getsize(path) > 0:
        df = pd.read_csv(path, delim_whitespace=True, header=None)
        df.columns = columns
    else:
        df = pd.DataFrame(columns=columns)  # Assign columns even if empty.
    dfs[key] = df

# Extract failure occurrence time and energy arrays.
t = {}
E = {}
# In case we would like to filter events in energy we also produce a boolean mask to filter events with energy greater than a certain threshold.
# At the moment we consider all events regardless of their energy.
# Indeed the relation between failure energy and outburst energy in photons is not straighfarward as most of the energy can be also released in the form of neutrino emission.
# Moreover also the less energetic events might produce short flares that might be detectable.
mask_E = {}

for key, df in dfs.items():
    t[key] = df['time'].values
    E[key] = df['energy'].values
    mask_E[key] = E[key] > 0
    t[key] = t[key][mask_E[key]]
    E[key] = E[key][mask_E[key]]

In [ ]:
# Define an array with the log10 of the initial magnetic field values for the different failure event sets.
log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

# Define initial magnetic fields where to evaluate the interpolated failure rate curves.
# Note that this is needed now to set the right colors.
log_B0_eval = np.linspace(11.0, 16.0, 100)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

Plot the failure rate from the simulation as a function of time and colorcode them according to their initial magnetic field strength.

In [ ]:
# Define bin edges
t_edges = np.linspace(0.0, 1.e6, 10001)
bin_widths = np.diff(t_edges)
bin_centers = t_edges[:-1] + bin_widths / 2

# Compute the histogram.
counts_1e12, _ = np.histogram(t['1e12'], bins=t_edges)
counts_1e13, _ = np.histogram(t['1e13'], bins=t_edges)
counts_1e14, _ = np.histogram(t['1e14'], bins=t_edges)
counts_1e15, _ = np.histogram(t['1e15'], bins=t_edges)
counts_5e15, _ = np.histogram(t['5e15'], bins=t_edges)

# Divide by bin width to find a rate.
rate_1e12 = counts_1e12 / bin_widths 
rate_1e13 = counts_1e13 / bin_widths 
rate_1e14 = counts_1e14 / bin_widths 
rate_1e15 = counts_1e15 / bin_widths 
rate_5e15 = counts_5e15 / bin_widths

# Plot using ax.step
fig, ax = plt.subplots(figsize=(15, 8))
ax.step(
    bin_centers,
    rate_1e12,
    where='post',
    color=cmap(norm(log_B0[0])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_1e13,
    where='post',
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_1e14,
    where='post',
    color=cmap(norm(log_B0[2])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_1e15,
    where='post',
    color=cmap(norm(log_B0[3])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_5e15,
    where='post',
    color=cmap(norm(log_B0[4])),
    rasterized=True,
    lw=4,
    alpha=1,
)



sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")

#ax.set_xlim(0.0, 1000000)
ax.set_ylim(0.0001, 2000)

Interpolate the failure rate to smooth out the curves and compare them with the corresponding analytical fit (the parameter of the fit are chosen by eye to resamble the trend of the curves).

In [ ]:
time_grid = np.logspace(0., 7., 50)

In [ ]:
fr1e12_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e12, k=1
)
fr1e13_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e13, k=1
)
fr1e14_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e14, k=1
)
fr1e15_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e15, k=1
)
fr5e15_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_5e15, k=1
)

rate_1e12_smooth = fr1e12_interpolator(time_grid)
rate_1e13_smooth = fr1e13_interpolator(time_grid)
rate_1e14_smooth = fr1e14_interpolator(time_grid)
rate_1e15_smooth = fr1e15_interpolator(time_grid)
rate_5e15_smooth = fr5e15_interpolator(time_grid)

In [ ]:
if (cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy") or (cfg["magneto-thermal_model"] == "BSk24_dip-tor_light"):
    failure_rate_fit_1e12 = failure_rate_fit(1.e-2, time_grid, 1.e0, -2.0)
    failure_rate_fit_1e13 = failure_rate_fit(1.e-1, time_grid, 1.e1, -2.0)
    failure_rate_fit_1e14 = failure_rate_fit(1.e0, time_grid, 1.e2, -2.0)
    failure_rate_fit_1e15 = failure_rate_fit(1.e1, time_grid, 1.e3, -2.0)
    failure_rate_fit_5e15 = failure_rate_fit(1.e2, time_grid, 2.e3, -2.0)
elif (cfg["magneto-thermal_model"] == "BSk24_multi_heavy") or (cfg["magneto-thermal_model"] == "BSk24_multi_light"):
    failure_rate_fit_1e12 = failure_rate_fit(1.e-1, time_grid, 1.e0, -2.0)
    failure_rate_fit_1e13 = failure_rate_fit(1.e0, time_grid, 1.e1, -2.0)
    failure_rate_fit_1e14 = failure_rate_fit(2.e1, time_grid, 2.e2, -2.0)
    failure_rate_fit_1e15 = failure_rate_fit(4.e2, time_grid, 1.e3, -2.0)
    failure_rate_fit_5e15 = failure_rate_fit(4.e3, time_grid, 2.e3, -2.0)

In [ ]:
# Plot using ax.step
fig, ax = plt.subplots(figsize=(15, 8))
ax.plot(
    time_grid,
    rate_1e12_smooth,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_1e13_smooth,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_1e14_smooth,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_1e15_smooth,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_5e15_smooth,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
    lw=4,
    alpha=1,
)

ax.plot(
    time_grid,
    failure_rate_fit_1e12,
    lw=4,
    ls=":",
    alpha=1,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    failure_rate_fit_1e13,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    ls=":",
    alpha=1,
)
ax.plot(
    time_grid,
    failure_rate_fit_1e14,
    lw=4,
    ls=":",
    alpha=1,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    time_grid,
    failure_rate_fit_1e15,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
    lw=4,
    ls=":",
    alpha=1,
)
ax.plot(
    time_grid,
    failure_rate_fit_5e15,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
    lw=4,
    ls=":",
    alpha=1,
)

# Create a legend for the line styles.
styleandles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=":",
        linewidth=4,
        label="Fit",
    ),
]
plt.legend(handles=styleandles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")

#ax.set_xlim(0.0, 1000000)
ax.set_ylim(0.0001, 2000)

In [ ]:
# When the event rate falls to less than 10^-2 in the simulations we stick the expected evolution from the analytical fit.
rate_1e12_smooth[rate_1e12_smooth <= 1.e-2] = failure_rate_fit_1e12[rate_1e12_smooth <= 1.e-2]
rate_1e13_smooth[rate_1e13_smooth <= 1.e-2] = failure_rate_fit_1e13[rate_1e13_smooth <= 1.e-2]
rate_1e14_smooth[rate_1e14_smooth <= 1.e-2] = failure_rate_fit_1e14[rate_1e14_smooth <= 1.e-2]
rate_1e15_smooth[rate_1e15_smooth <= 1.e-2] = failure_rate_fit_1e15[rate_1e15_smooth <= 1.e-2]
rate_5e15_smooth[rate_5e15_smooth <= 1.e-2] = failure_rate_fit_5e15[rate_5e15_smooth <= 1.e-2]

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    time_grid,
    rate_1e12_smooth,
    lw=4,
    alpha=1,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e13_smooth,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_1e14_smooth,
    lw=4,
    alpha=1,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e15_smooth,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.plot(
    time_grid,
    rate_5e15_smooth,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
    lw=4,
    alpha=1,
)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")
#ax.set_xlim(0.0, 1000000)
ax.set_ylim(0.000001, 2000)

## Construct the interpolator function

In [ ]:
# Create a grid of initial magnetic fields and stack together all the cooling curves.
B0 = 10**log_B0
rate_t_stack = np.vstack(
    (rate_1e12_smooth, rate_1e13_smooth, rate_1e14_smooth, rate_1e15_smooth, rate_5e15_smooth)
).T

# Define the minimum and maximum age in [yr] and the minimum and maximum initial magnetic field in [G].
time_range = np.array([0.0, 1.0e7])
B0_range = np.array([1.0e11, 1.0e16])

failure_rate_interpolator = interpolate.RectBivariateSpline(
    time_grid,
    B0,
    rate_t_stack,
    bbox=[time_range[0], time_range[1], B0_range[0], B0_range[1]],
    kx=1,
    ky=1,
)

In [ ]:
# Save the interpolator function and try to import it again to see if it works.
interpolator_path = base_path.joinpath(cfg["magneto-thermal_path"], "interpolator_crust_failure_rate.pkl")
with open(
    interpolator_path,
    "wb",
) as f:
    pickle.dump(failure_rate_interpolator, f)

with open(
    interpolator_path,
    "rb",
) as f:
    failure_rate_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of times and initial magnetic fields at which we evaluate the interpolated cooling curves.
t_eval = np.logspace(0.0, 7, 500)
B0_eval = 10**log_B0_eval

failure_rate_interp = failure_rate_interpolator_import(t_eval, B0_eval)
print(failure_rate_interp.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e-5, 2.0e3)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel("Failures per year")

ax.plot(
    time_grid,
    rate_1e12_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e13_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e14_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e15_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_5e15_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

for i in range(len(B0_eval)):
    ax.plot(
        t_eval,
        failure_rate_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )

# Create a legend for the line styles.
styleandles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=4,
        label="Interpolated",
    ),
]
plt.legend(handles=styleandles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

In [ ]:
# Try to evaluate the failure rate for a set of random values of the age and the initial magnetic field.
t_eval_test = np.array([1.0e3, 1.0e2])
B0_eval_test = np.logspace(13.0, 15, 2)

failure_rate_interp_test = failure_rate_interpolator_import.ev(t_eval_test, B0_eval_test)
print(failure_rate_interp_test)

Plot the released magnetic energy during the failures as a function of time.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    t["5e15"],
    E["5e15"],
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[4])),
    markersize=6,
    alpha=0.2,
    rasterized=True,
)
ax.plot(
    t["1e15"],
    E["1e15"],
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[3])),
    markersize=6,
    alpha=0.2,
    rasterized=True,
)
ax.plot(
    t["1e14"],
    E["1e14"],
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[2])),
    markersize=6,
    alpha=0.2,
    rasterized=True,
)
ax.plot(
    t["1e13"],
    E["1e13"],
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[1])),
    markersize=6,
    alpha=0.2,
    rasterized=True,
)
ax.plot(
    t["1e12"],
    E["1e12"],
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[0])),
    markersize=6,
    alpha=0.2,
    rasterized=True,
)

ax.set_xlabel("Time [yr]")
ax.set_ylabel("Crustal failure energy [erg]")
ax.set_xscale("log")
ax.set_yscale("log")

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

## 